In [1]:
# Import required libraries and dependencies
import pandas as pd
import hvplot.pandas
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

In [3]:
# Load the data into a Pandas DataFrame
df_market_data = pd.read_csv(
    "Resources/crypto_market_data.csv",
    index_col="coin_id")

# Display sample data
df_market_data.head(10)

,price_change_percentage_24h,price_change_percentage_7d,price_change_percentage_14d,price_change_percentage_30d,price_change_percentage_60d,price_change_percentage_200d,price_change_percentage_1y
coin_id,,,,,,,
bitcoin,1.08388,7.60278,6.57509,7.67258,-3.25185,83.51840,37.51761
ethereum,0.22392,10.38134,4.80849,0.13169,-12.88890,186.77418,101.96023
tether,-0.21173,0.04935,0.00640,-0.04237,0.28037,-0.00542,0.01954
ripple,-0.37819,-0.60926,2.24984,0.23455,-17.55245,39.53888,-16.60193
bitcoin-cash,2.90585,17.09717,14.75334,15.74903,-13.71793,21.66042,14.49384
binancecoin,2.10423,12.85511,6.80688,0.05865,36.33486,155.61937,69.69195
chainlink,-0.23935,20.69459,9.30098,-11.21747,-43.69522,403.22917,325.13186
cardano,0.00322,13.99302,5.55476,10.10553,-22.84776,264.51418,156.09756
litecoin,-0.06341,6.60221,7.28931,1.21662,-17.23960,27.49919,-12.66408


In [4]:
# Generate summary statistics
df_market_data.describe()

,price_change_percentage_24h,price_change_percentage_7d,price_change_percentage_14d,price_change_percentage_30d,price_change_percentage_60d,price_change_percentage_200d,price_change_percentage_1y
count,41.000000,41.000000,41.000000,41.000000,41.000000,41.000000,41.000000
mean,-0.269686,4.497147,0.185787,1.545693,-0.094119,236.537432,347.667956
std,2.694793,6.375218,8.376939,26.344218,47.365803,435.225304,1247.842884
min,-13.527860,-6.094560,-18.158900,-34.705480,-44.822480,-0.392100,-17.567530
25%,-0.608970,0.047260,-5.026620,-10.438470,-25.907990,21.660420,0.406170
50%,-0.063410,3.296410,0.109740,-0.042370,-7.544550,83.905200,69.691950
75%,0.612090,7.602780,5.510740,4.578130,0.657260,216.177610,168.372510
max,4.840330,20.694590,24.239190,140.795700,223.064370,2227.927820,7852.089700


In [5]:
# Plot your data to see what's in your DataFrame
df_market_data.hvplot.line(
    width=800,
    height=400,
    rot=90
)

:NdOverlay   [Variable]
   :Curve   [coin_id]   (value)

---

### Prepare the Data

In [10]:
# Use the `StandardScaler()` module from scikit-learn to normalize the data from the CSV file
market_data_scaled = StandardScaler().fit_transform(
    df_market_data[["price_change_percentage_24h", "price_change_percentage_7d", "price_change_percentage_14d", "price_change_percentage_30d", "price_change_percentage_60d"]]
)

In [12]:
# Create a DataFrame with the scaled data
df_market_scaled = pd.DataFrame(
    market_data_scaled,
    columns=["price_change_percentage_24h", "price_change_percentage_7d", "price_change_percentage_14d", "price_change_percentage_30d", "price_change_percentage_60d"]
)
# Copy the crypto names from the original DataFrame
df_market_scaled["coin_id"] = df_market_data.index

# Set the coin_id column as index
df_market_scaled = df_market_scaled.set_index("coin_id")

# Display the scaled DataFrame
df_market_scaled.head(5)

,price_change_percentage_24h,price_change_percentage_7d,price_change_percentage_14d,price_change_percentage_30d,price_change_percentage_60d
coin_id,,,,,
bitcoin,0.508529,0.493193,0.772200,0.235460,-0.067495
ethereum,0.185446,0.934445,0.558692,-0.054341,-0.273483
tether,0.021774,-0.706337,-0.021680,-0.061030,0.008005
ripple,-0.040764,-0.810928,0.249458,-0.050388,-0.373164
bitcoin-cash,1.193036,2.000959,1.760610,0.545842,-0.291203


---

### Find the Best Value for k Using the Original Scaled DataFrame.

In [83]:
# Create a list with the number of k-values from 1 to 11
k = list(range(1, 12))

In [84]:
# Create an empty list to store the inertia values
inertia = []
# Create a for loop to compute the inertia with each possible value of k
# Inside the loop:
# 1. Create a KMeans model using the loop counter for the n_clusters
# 2. Fit the model to the data using `df_market_data_scaled`
# 3. Append the model.inertia_ to the inertia list
for i in k:
    model = KMeans(n_clusters=i, random_state=42)
    model.fit(df_market_scaled)
    inertia.append(model.inertia_)

In [85]:
# Create a dictionary with the data to plot the Elbow curve
elbow_data_pca = {
    "k": k,
    "inertia": inertia
}

# Create a DataFrame with the data to plot the Elbow curve
df_elbow_pca = pd.DataFrame(elbow_data_pca)

In [86]:
# Plot a line chart with all the inertia values computed with
# the different values of k to visually identify the optimal value for k.
elbow_plot_pca = df_elbow_pca.hvplot.line(x="k", y="inertia", title="Elbow Curve Using PCA Data", xticks=k)
elbow_plot_pca

:Curve   [k]   (inertia)

#### Answer the following question: 

**Question:** What is the best value for `k`?

**Answer:** K=4. This is the point where the rate of decrease sharply changes

---

### Cluster Cryptocurrencies with K-means Using the Original Scaled DataFrame

In [51]:
# Initialize the K-Means model using the best value for k
kmeans_model = KMeans(n_clusters=4, random_state=42)

In [52]:
# Fit the K-Means model using the scaled DataFrame
kmeans_model.fit(df_market_scaled)


KMeans(n_clusters=4, random_state=42)

In [53]:
# Predict the clusters to group the cryptocurrencies using the scaled DataFrame
predicted_clusters = kmeans_model.predict(df_market_scaled.drop(columns=["Cluster"], errors="ignore"))

# Print the resulting array of cluster values.
df_market_scaled.head()

,price_change_percentage_24h,price_change_percentage_7d,price_change_percentage_14d,price_change_percentage_30d,price_change_percentage_60d
coin_id,,,,,
bitcoin,0.508529,0.493193,0.772200,0.235460,-0.067495
ethereum,0.185446,0.934445,0.558692,-0.054341,-0.273483
tether,0.021774,-0.706337,-0.021680,-0.061030,0.008005
ripple,-0.040764,-0.810928,0.249458,-0.050388,-0.373164
bitcoin-cash,1.193036,2.000959,1.760610,0.545842,-0.291203


In [54]:
# Create a copy of the scaled DataFrame
df_scaled_copy = df_market_scaled.copy()

In [55]:
# Add a new column to the copy of the scaled DataFrame with the predicted clusters
df_scaled_copy["Predicted_Cluster"] = predicted_clusters

# Display the copy of the scaled DataFrame
df_scaled_copy.head()

,price_change_percentage_24h,price_change_percentage_7d,price_change_percentage_14d,price_change_percentage_30d,price_change_percentage_60d,Predicted_Cluster
coin_id,,,,,,
bitcoin,0.508529,0.493193,0.772200,0.235460,-0.067495,0
ethereum,0.185446,0.934445,0.558692,-0.054341,-0.273483,0
tether,0.021774,-0.706337,-0.021680,-0.061030,0.008005,2
ripple,-0.040764,-0.810928,0.249458,-0.050388,-0.373164,2
bitcoin-cash,1.193036,2.000959,1.760610,0.545842,-0.291203,0


In [87]:
# Create a scatter plot using hvPlot by setting
# `x="price_change_percentage_24h"` and `y="price_change_percentage_7d"`.
# Color the graph points with the labels found using K-Means and
# add the crypto name in the `hover_cols` parameter to identify
# the cryptocurrency represented by each data point.
scatter_plot = df_scaled_copy.hvplot.scatter(
    x="price_change_percentage_24h",
    y="price_change_percentage_7d",
    c="Predicted_Cluster",
    hover_cols=["coin_id"],       # Show coin name on hover
    title="Crypto Clustering by 24h and 7d Price Change",
    xlabel="24h % Change",
    ylabel="7d % Change",
    height=500,
    width=700
)
scatter_plot

:Scatter   [price_change_percentage_24h]   (price_change_percentage_7d,Predicted_Cluster,coin_id)

---

### Optimize Clusters with Principal Component Analysis.

In [57]:
# Create a PCA model instance and set `n_components=3`.
pca = PCA(n_components=3)

In [59]:
# Use the PCA model with `fit_transform` to reduce the original scaled DataFrame
# down to three principal components.

features = [
    "price_change_percentage_24h",
    "price_change_percentage_7d",
    "price_change_percentage_14d",
    "price_change_percentage_30d",
    "price_change_percentage_60d"
]
market_pca_data = pca.fit_transform(df_market_scaled[features])
df_pca = pd.DataFrame(
    market_pca_data,
    columns=["PC1", "PC2", "PC3"],
    index=df_market_scaled.index )

# View the scaled PCA data
df_pca.head()

,PC1,PC2,PC3
coin_id,,,
bitcoin,0.795104,0.662906,0.153885
ethereum,0.413769,1.047773,-0.157623
tether,-0.195508,-0.518261,0.201535
ripple,-0.260748,-0.340601,0.145741
bitcoin-cash,1.961019,2.239724,0.182981


In [32]:
# Retrieve the explained variance to determine how much information
# can be attributed to each principal component.
explained_variance = pca.explained_variance_ratio_
for i, variance in enumerate(explained_variance, start=1):
    print(f"PC{i} explains {variance:.2%} of the variance")

PC1 explains 47.86% of the variance
PC2 explains 26.61% of the variance
PC3 explains 16.85% of the variance


In [40]:
# Create a new DataFrame with the PCA data.
df_pca = pd.DataFrame(
    market_pca_data,                          
    columns=["PC1", "PC2", "PC3"],    
    index=df_market_scaled.index      
)

# Display the scaled PCA DataFrame
df_pca.head()

,PC1,PC2,PC3
coin_id,,,
bitcoin,0.795104,0.662906,0.153885
ethereum,0.413769,1.047773,-0.157623
tether,-0.195508,-0.518261,0.201535
ripple,-0.260748,-0.340601,0.145741
bitcoin-cash,1.961019,2.239724,0.182981


---

### Find the Best Value for k Using the Scaled PCA DataFrame

In [104]:
# Create a list with the number of k-values from 1 to 11
k_values = list(range(1, 12))

In [109]:
# Create an empty list to store the inertia values
inertia_values = []

# Create a for loop to compute the inertia with each possible value of k
# Inside the loop:
# 1. Create a KMeans model using the loop counter for the n_clusters
# 2. Fit the model to the data using `df_market_data_pca`
# 3. Append the model.inertia_ to the inertia list
for k in k_values:
    model = KMeans(n_clusters=i, random_state=42)
    model.fit(market_pca_data)
    inertia_values.append(model.inertia_)

In [110]:
# Create a dictionary with the data to plot the Elbow curve
elbow_data_pca = {
    "k": k_values,
    "inertia": inertia_values
}
# Create a DataFrame with the data to plot the Elbow curve
df_elbow_pca = pd.DataFrame(elbow_data_pca)
df_elbow_pca.head()

,k,inertia
0,1,16.192363
1,2,16.192363
2,3,16.192363
3,4,16.192363
4,5,16.192363


In [111]:
# Plot a line chart with all the inertia values computed with
# the different values of k to visually identify the optimal value for k.
elbow_plot_pca = df_elbow_pca.hvplot.line(
    x="k",
    y="inertia",
    title="Elbow Curve Using Scaled PCA Data",
    xlabel="Number of Clusters (k)",
    ylabel="Inertia",
    line_width=2,
    height=400,
    width=600
)

elbow_plot_pca

:Curve   [k]   (inertia)

#### Answer the following questions: 

* **Question:** What is the best value for `k` when using the PCA data?

  * **Answer:** k=4


* **Question:** Does it differ from the best k value found using the original data?

  * **Answer:** No. the best k value found using the original scaled data is also 4, which confirms that the clustering structure is preserved even dimensionality reduction with PCA

### Cluster Cryptocurrencies with K-means Using the Scaled PCA DataFrame

In [112]:
# Initialize the K-Means model using the best value for k
kmeans_pca = KMeans(n_clusters=4, random_state=42)

In [113]:
# Fit the K-Means model using the PCA data
kmeans_pca.fit(df_pca[["PC1", "PC2", "PC3"]])

KMeans(n_clusters=4, random_state=42)

In [114]:
# Predict the clusters to group the cryptocurrencies using the scaled PCA DataFrame
cluster_labels = kmeans_pca.predict(df_pca[["PC1", "PC2", "PC3"]])

# Print the resulting array of cluster values.
print(cluster_labels)

[0 0 2 2 0 0 0 0 0 2 2 2 2 0 2 0 2 2 0 2 2 0 2 2 2 2 2 2 0 2 2 2 3 0 2 2 1
 2 2 2 2]


In [117]:
# Create a copy of the scaled PCA DataFrame
df_pca_scaled_copy = df_pca.copy()

# Add a new column to the copy of the PCA DataFrame with the predicted clusters
df_pca_scaled_copy["Predicted_Cluster"] = cluster_labels

# Display the copy of the scaled PCA DataFrame
df_pca_scaled_copy.head()

,PC1,PC2,PC3,Predicted_Cluster
coin_id,,,,
bitcoin,0.795104,0.662906,0.153885,0
ethereum,0.413769,1.047773,-0.157623,0
tether,-0.195508,-0.518261,0.201535,2
ripple,-0.260748,-0.340601,0.145741,2
bitcoin-cash,1.961019,2.239724,0.182981,0


In [118]:
# Create a scatter plot using hvPlot by setting
# `x="PC1"` and `y="PC2"`.
# Color the graph points with the labels found using K-Means and
# add the crypto name in the `hover_cols` parameter to identify
# the cryptocurrency represented by each data point.
scatter_plot = df_pca_scaled_copy.hvplot.scatter(
    x="PC1",
    y="PC2",
    c="Predicted_Cluster",           
    colormap="Category10",           
    hover_cols=["coin_id"],          
    title="Cryptocurrency Clusters (PC1 vs PC2)",
    xlabel="Principal Component 1",
    ylabel="Principal Component 2",
    height=500,
    width=700
)

scatter_plot

:Scatter   [PC1]   (PC2,Predicted_Cluster,coin_id)

### Visualize and Compare the Results

In this section, you will visually analyze the cluster analysis results by contrasting the outcome with and without using the optimization techniques.

In [129]:
# Composite plot to contrast the Elbow curves
df_elbow_scaled = pd.DataFrame({
    "k": k_values,
    "inertia": inertia_values  # ← your original scaled data inertia list
})
df_elbow_scaled["Source"] = "Original Scaled Data" 
df_elbow_pca["Source"] = "PCA Reduced Data"
df_elbow_combined = pd.concat([df_elbow_scaled, df_elbow_pca], ignore_index=True)
composite_elbow_plot = df_elbow_combined.hvplot.line(
    x="k",
    y="inertia",
    by="Source",  
    title="Comparison of Elbow Curves: Scaled vs PCA Data",
    xlabel="Number of Clusters (k)",
    ylabel="Inertia",
    line_width=2,
    height=500,
    width=700,
    legend="top_left"
)
composite_elbow_plot

:NdOverlay   [Source]
   :Curve   [k]   (inertia)

In [135]:
# Composite plot to contrast the clusters
df_scaled_copy["Source"] = "Original Scaled Data"
df_pca_scaled_copy["Source"] = "PCA Reduced Data"
df_cluster_combined = pd.concat([df_scaled_copy, df_pca_scaled_copy], ignore_index=True)
cluster_composite_plot = df_cluster_combined.hvplot.scatter(
    x="PC1",
    y="PC2",
    c="Cluster",
    by="Source",         
    hover_cols=["coin_id"],
    col="Source",        
    title="Cluster Comparison: Original Scaled vs PCA Reduced",
    height=400,
    width=500
)
cluster_composite_plot

:GridSpace   [Source]
   :NdOverlay   [Source]
      :Scatter   [PC1]   (PC2)

#### Answer the following question: 

  * **Question:** After visually analyzing the cluster analysis results, what is the impact of using fewer features to cluster the data using K-Means?

  * **Answer:** Using fewer features through PCA typically improves efficiency, reduces noise, and helps K-Means find clearer, more robust clusters — at the cost of interpretability and possibly overlooking minor feature-specific patterns.